In [4]:
:dep reqwest = { version = "0.11", features = ["blocking"] }
:dep scraper = "0.17"
:dep serde_json = "1"

In [18]:
use reqwest::blocking::get;
use scraper::{Html, Selector};
use std::process::Command;
use std::fs;

let url = "https://ksnctf.sweetduet.info/q/3/unya.html";
let html = get(url)
    .expect("URLにアクセスできませんでした")
    .text()
    .expect("テキスト読み込み失敗");

let document = Html::parse_document(&html);
let selector = Selector::parse("script").unwrap();

let unya_code = document
    .select(&selector)
    .nth(1)
    .expect("scriptタグが見つかりません")
    .inner_html();
let unya_code = unya_code.trim().to_string();

let wrapper_js = format!(r#"
var captured_p = null;
var alert = function(msg) {{}};
var $ = function(arg) {{
    if (typeof arg === 'function') {{
        arg();
    }}
    return {{
        submit: function(fn) {{
            fn();
        }},
        val: function() {{ return ""; }},
    }};
}};
var OriginalArray = Array;
Array = function() {{
    var result = OriginalArray.apply(this, arguments);
    if (arguments.length === 21) {{
        captured_p = Array.from(arguments);
    }}
    return result;
}};
Array.from = OriginalArray.from;
eval({unya_json});
if (captured_p) {{
    var flag = "";
    for (var i = 0; i < captured_p.length; i++) {{
        flag += String.fromCharCode(captured_p[i] / (i + 1));
    }}
    process.stdout.write(flag + "\n");
}} else {{
    process.stdout.write("ERROR: pを捕捉できませんでした\n");
}}
"#,
    unya_json = serde_json::to_string(&unya_code).unwrap()
);

let output = Command::new("node")
    .arg("-e")
    .arg(&wrapper_js)
    .stdout(std::process::Stdio::piped())
    .stderr(std::process::Stdio::piped())
    .spawn()
    .expect("nodeコマンドの起動失敗")
    .wait_with_output()
    .expect("node実行待機失敗");

if output.status.success() {
    let flag = String::from_utf8_lossy(&output.stdout).trim().to_string();
    fs::write("flag.txt", &flag).expect("flag.txtへの書き込み失敗");
    println!("flag.txtに保存しました");
} else {
    println!("エラー: {}", String::from_utf8_lossy(&output.stderr));
}

flag.txtに保存しました


()

# ksnctf Problem 3 「Crawling Chaos」 Writeup (Rust)

## 問題概要

`http://ksnctf.sweetduet.info/q/3/unya.html` にアクセスすると、テキストフォームが1つあるWebページが表示される。何を入力して Submit しても "No" というアラートが出るだけ。ページのソースを見ると `<script>` タグ内に **unyaencode** で難読化された JavaScript が埋め込まれている。

```
(ᒧᆞωᆞ)=(/ᆞωᆞ/),(ᒧᆞωᆞ).ᒧうー=-!!(/ᆞωᆞ/).にゃー, ...（以下延々と続く）
```

---

## 解法の方針

unyaencode は JavaScript のランタイム（RegExp オブジェクトへの動的プロパティ付与など）に強く依存しており、Rust の JS エンジン（Boa など）では完全な再現が困難。そこで以下の方針を採用した。

1. Rust で対象 URL から HTML を取得し、unyaencode コードを抽出する
2. `$`・`alert` をモックした JS ラッパーを組み立て、**Node.js** に渡して実行する
3. `Array()` コンストラクタを横取りして `p` 配列を捕捉し、フラグを逆算する
4. 結果を `flag.txt` に保存する

---

## 詰まったポイントと解決策

### 1. Boa エンジンで `TypeError: not a callable function`

unyaencode は内部で `$`（jQuery）を呼び出す。Boa は標準的な JS しか実行できず、DOM 依存の処理は動かない。**デコード済みロジックを直接 Rust に移植する**か、本物の JS エンジンを使う必要がある。

### 2. `trim()` の non-static lifetime エラー

evcxr はセル間で変数を保持する仕組み上、参照（借用）を持つ変数を保持できない。`trim()` は `&str`（参照）を返すため、`.to_string()` で所有権のある `String` に変換する必要がある。

### 3. `format!` 内のコメントで `{}` のエスケープエラー

`format!` のraw文字列内では `{` と `}` は特殊文字。JS コメント中の `{}` もエスケープ対象になるためエラーになる。**コメントを削除する**ことで回避。

### 4. `alert is not defined`

デコード済み JS が `alert()` を呼ぶが、Node.js にはブラウザの `alert` がない。`var alert = function(msg) {};` をモックとして追加することで解決。

---

## 実行環境の要件

このコードは以下の環境で動作する。

- **Rust** （evcxr_jupyter カーネル経由で Jupyter Notebook 上で実行）
- **Node.js がインストールされていること**

Rust から `std::process::Command` で `node -e` を呼び出しているため、**Node.js が PATH に通っている必要がある**。インストール済みかどうかは以下で確認できる。

```rust
let check = Command::new("node").arg("--version").output().unwrap();
println!("{}", String::from_utf8_lossy(&check.stdout));
// v20.x.x などが表示されれば OK
```

Node.js が未インストールの場合は https://nodejs.org から公式インストーラーをダウンロードしてインストールする。

